# Requirement 11: Unique ephemeral secrets

> Generate a new cryptographically random secret for every session.

**Covered by:** Caller

**How it is fulfilled:** The orchestrator generates a new 256-bit key from the operating system's cryptographic random source for every session. Nothing is reused.

In [1]:
import sys
sys.path.insert(0, "..")

In [2]:
from helpers import *
a, sid_a = new_session()
b, sid_b = new_session()
key_a = install_session_key(a, sid_a)
key_b = install_session_key(b, sid_b)

In [3]:
check("each key is 256 bits", len(key_a) == 32 and len(key_b) == 32)
check("the two sessions got different keys", key_a != key_b)
check("a key from one session does not work in the other",
      "REJECTED" in send(b, signed_request(key_a, sid_b, 1, [1])))
check("its own key works", send(b, signed_request(key_b, sid_b, 2, [1])).startswith("OK"))

PASS  each key is 256 bits
PASS  the two sessions got different keys


PASS  a key from one session does not work in the other


PASS  its own key works


In [4]:
cleanup(a, b)

**Caveat:** This is the orchestrator's responsibility; the platform does not enforce it. The demo uses `os.urandom`; production would use the same or a managed key service.